# gen-gec-errant Pipeline — Colab

Run the full **generate → GEC → ERRANT → analysis** pipeline for a single model.

**Data source:** `phd-experimental-data/cefr-classification/data/splits/`  
**Models:** `_p/artificial-learners/models/` (fine-tuned) or HuggingFace Hub (native)  
**Output:** `_p/artificial-learners/generation-gec-errant/results/`  

**Runtime:** Depends on model size and dataset limits. ~10–30 min per dataset on T4 GPU.

*Last updated: 2026-03-16*

## 0. Mount Google Drive & Install

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
%%time
# Install core dependencies
!pip install torch transformers errant spacy numpy scipy matplotlib pandas pyyaml accelerate -q
!python -m spacy download en_core_web_sm -q

In [ ]:
import sys
from pathlib import Path

# ── Install gen_gec_errant from Google Drive ──────────────────────────
# Option A: Install from GDrive copy of the repo
GDRIVE = Path('/content/drive/MyDrive')
REPO_ON_DRIVE = GDRIVE / '_p/artificial-learners/generation-gec-errant'

# Option B: Clone to local /content and install (uncomment if repo is on GitHub)
# !git clone https://github.com/YOUR_USER/gen-gec-errant.git /content/gen-gec-errant
# REPO_ON_DRIVE = Path('/content/gen-gec-errant')

if REPO_ON_DRIVE.exists():
    # Copy to local filesystem for faster imports
    !cp -r "{REPO_ON_DRIVE}" /content/gen-gec-errant
    !pip install -e /content/gen-gec-errant -q
    print(f'Installed gen_gec_errant from {REPO_ON_DRIVE}')
else:
    print(f'WARNING: Repo not found at {REPO_ON_DRIVE}')
    print('Please either:')
    print('  1. Upload the repo to Google Drive at _p/artificial-learners/generation-gec-errant/')
    print('  2. Or uncomment the git clone option above')
    sys.exit(1)

## 1. Configuration

Select a model from the registry. Set `MODEL_KEY` to one of the available models below.

In [ ]:
from pathlib import Path
from datetime import datetime
import torch

# ── Paths ──────────────────────────────────────────────────────────────
GDRIVE = Path('/content/drive/MyDrive')

DATA_ROOT = GDRIVE / 'phd-experimental-data/cefr-classification/data/splits'
MODELS_ROOT = GDRIVE / '_p/artificial-learners/models'
OUTPUT_ROOT = GDRIVE / '_p/artificial-learners/generation-gec-errant/results'

TIMESTAMP = datetime.now().strftime('%Y%m%d-%H%M%S')

# ── Data files ─────────────────────────────────────────────────────────
# Each entry: (path, description)
# Same splits used across the project (cf. colab_train_bert.ipynb)
DATASETS = {
    'norm-CELVA-SP': {
        'path': DATA_ROOT / 'norm-CELVA-SP.csv',
        'description': 'Spanish L1 learner English (primary)',
    },
    'norm-EFCAMDAT-test': {
        'path': DATA_ROOT / 'norm-EFCAMDAT-test.csv',
        'description': 'In-domain learner English',
    },
    'norm-KUPA-KEYS': {
        'path': DATA_ROOT / 'norm-KUPA-KEYS.csv',
        'description': 'Cross-corpus learner English',
    },
}

# ── Model Registry ─────────────────────────────────────────────────────
# Maps model key -> {params, base_hf_id, model_family, gdrive_subpath, description}
# For fine-tuned models, gdrive_subpath points to the directory containing final/
# For native models, hf_model_id is used directly from HuggingFace Hub

MODEL_REGISTRY = {
    # ── Native (pre-trained) baselines ──
    'gpt2-small-native': {
        'params': '124M',
        'hf_model_id': 'gpt2',
        'model_family': 'gpt2',
        'gdrive_subpath': None,
        'is_learner_tuned': False,
        'description': 'GPT-2 Small native (zero-shot baseline)',
    },
    # ── GPT-2 fine-tuned ──
    'ft-gpt2-small': {
        'params': '124M',
        'hf_model_id': None,
        'model_family': 'gpt2',
        'gdrive_subpath': '2026-02-23-model/gpt2-small-all-data-resume',
        'is_learner_tuned': True,
        'description': 'GPT-2 Small (124M) fine-tuned on EFCAMDAT all-data',
    },
    'ft-gpt2-medium': {
        'params': '355M',
        'hf_model_id': None,
        'model_family': 'gpt2',
        'gdrive_subpath': '2026-02-23-model/gpt2-medium-all-data-resume',
        'is_learner_tuned': True,
        'description': 'GPT-2 Medium (355M) fine-tuned on EFCAMDAT all-data',
    },
    'ft-gpt2-large': {
        'params': '774M',
        'hf_model_id': None,
        'model_family': 'gpt2',
        'gdrive_subpath': '2026-02-20-model/gpt2-large-all-data-20260222-092036',
        'is_learner_tuned': True,
        'description': 'GPT-2 Large (774M) fine-tuned on EFCAMDAT all-data',
    },
    # ── Pythia fine-tuned ──
    'ft-pythia-70m': {
        'params': '70M',
        'hf_model_id': None,
        'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-70m-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 70M fine-tuned on EFCAMDAT all-data',
    },
    'ft-pythia-160m': {
        'params': '160M',
        'hf_model_id': None,
        'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-160m-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 160M fine-tuned on EFCAMDAT all-data',
    },
    'ft-pythia-410m': {
        'params': '410M',
        'hf_model_id': None,
        'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-410m-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 410M fine-tuned on EFCAMDAT all-data',
    },
    'ft-pythia-1b': {
        'params': '1B',
        'hf_model_id': None,
        'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-1b-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 1B fine-tuned on EFCAMDAT all-data',
    },
    'ft-pythia-1.4b': {
        'params': '1.4B',
        'hf_model_id': None,
        'model_family': 'pythia',
        'gdrive_subpath': 'pythia/pythia-1.4b-all-data',
        'is_learner_tuned': True,
        'description': 'Pythia 1.4B fine-tuned on EFCAMDAT all-data',
    },
    # ── SmolLM2 fine-tuned ──
    'ft-smollm2-135m': {
        'params': '135M',
        'hf_model_id': None,
        'model_family': 'smollm2',
        'gdrive_subpath': 'smollm2/smollm2-135m-all-data',
        'is_learner_tuned': True,
        'description': 'SmolLM2 135M fine-tuned on EFCAMDAT all-data',
    },
    'ft-smollm2-360m': {
        'params': '360M',
        'hf_model_id': None,
        'model_family': 'smollm2',
        'gdrive_subpath': 'smollm2/smollm2-360m-all-data',
        'is_learner_tuned': True,
        'description': 'SmolLM2 360M fine-tuned on EFCAMDAT all-data',
    },
    'ft-smollm2-1.7b': {
        'params': '1.7B',
        'hf_model_id': None,
        'model_family': 'smollm2',
        'gdrive_subpath': 'smollm2/smollm2-1.7b-all-data',
        'is_learner_tuned': True,
        'description': 'SmolLM2 1.7B fine-tuned on EFCAMDAT all-data',
    },
}

# ══════════════════════════════════════════════════════════════════════
#  SELECT YOUR MODEL HERE
# ══════════════════════════════════════════════════════════════════════
MODEL_KEY = 'ft-gpt2-small'  # <-- change this to run a different model
# ══════════════════════════════════════════════════════════════════════

# Sentence limits (set to None for full paper reproduction)
MAX_SENTENCES = None   # None = all data; set to 50 for quick test

# ── Pipeline params ────────────────────────────────────────────────────
GEC_MODEL = 'grammarly/coedit-large'
GEC_METHOD = 'dedicated'
BATCH_SIZE = 8          # GPU-optimized (was 2 for CPU)
GEC_BATCH_SIZE = 16     # GEC can handle larger batches
SEED = 42

# ── Resolve model path ─────────────────────────────────────────────────
model_info = MODEL_REGISTRY[MODEL_KEY]

if model_info['gdrive_subpath']:
    # Fine-tuned model on GDrive
    MODEL_PATH = str(MODELS_ROOT / model_info['gdrive_subpath'] / 'final')
else:
    # Native model from HuggingFace Hub
    MODEL_PATH = model_info['hf_model_id']

RUN_DIR = OUTPUT_ROOT / f"{MODEL_KEY}-{TIMESTAMP}"
RUN_DIR.mkdir(parents=True, exist_ok=True)

# ── Device ──────────────────────────────────────────────────────────────
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

print(f'\nModel:       {MODEL_KEY} ({model_info["params"]})')
print(f'Description: {model_info["description"]}')
print(f'Model path:  {MODEL_PATH}')
print(f'Output dir:  {RUN_DIR}')
print(f'Batch size:  {BATCH_SIZE}')
print(f'Max sent.:   {MAX_SENTENCES or "all"}')

print('\nDatasets:')
for name, info in DATASETS.items():
    exists = info['path'].exists()
    print(f'  {name}: {info["path"].name} (exists: {exists})')

## 1a. Copy Model to Local Disk (faster I/O)

Copying model weights from GDrive to Colab's local SSD avoids slow I/O during inference.

In [ ]:
import shutil

LOCAL_MODEL_PATH = MODEL_PATH  # default: use GDrive path directly

if model_info['gdrive_subpath']:
    gdrive_model_dir = Path(MODEL_PATH)
    local_cache = Path(f'/content/models/{MODEL_KEY}')

    if gdrive_model_dir.exists():
        if not local_cache.exists():
            print(f'Copying model to local disk for faster I/O...')
            print(f'  From: {gdrive_model_dir}')
            print(f'  To:   {local_cache}')
            shutil.copytree(gdrive_model_dir, local_cache)
            print(f'  Done. Size: {sum(f.stat().st_size for f in local_cache.rglob("*") if f.is_file()) / 1e9:.2f} GB')
        else:
            print(f'Model already cached locally: {local_cache}')
        LOCAL_MODEL_PATH = str(local_cache)
    else:
        print(f'WARNING: Model not found on GDrive: {gdrive_model_dir}')
        print('Make sure the model weights are uploaded to Google Drive.')
else:
    print(f'Using HuggingFace Hub model: {MODEL_PATH} (will download automatically)')

print(f'\nEffective model path: {LOCAL_MODEL_PATH}')

## 2. Run Pipeline Per Dataset

Runs the full 5-stage pipeline (data loading → generation → GEC → ERRANT annotation → analysis) for each dataset.

In [ ]:
import logging
import time
import yaml

logging.basicConfig(level=logging.INFO, format='%(name)s | %(message)s')

from gen_gec_errant.pipeline import run_pipeline
from gen_gec_errant.pipeline.config import load_config_from_yaml


def write_config_yaml(dataset_name, data_path, output_dir):
    """Write a YAML pipeline config for one dataset and return its path."""
    config_dir = RUN_DIR / 'configs'
    config_dir.mkdir(parents=True, exist_ok=True)
    config_path = config_dir / f'{dataset_name}.yaml'

    is_learner = model_info.get('is_learner_tuned', False)

    config = {
        'data_loader': {
            'data_path': str(data_path),
            'text_column': 'text',
            'max_sentences': MAX_SENTENCES,
            'min_words': 10,
            'max_words': 500,
            'prompt_ratio': 0.5,
            'min_prompt_words': 5,
        },
        'generation': {
            'max_new_tokens': 50,
            'min_new_tokens': 10,
            'temperature': 1.0,
            'top_k': 50,
            'top_p': 0.95,
            'do_sample': True,
            'repetition_penalty': 1.2,
        },
        'gec': {
            'method': GEC_METHOD,
            'model_id': GEC_MODEL,
            'batch_size': GEC_BATCH_SIZE,
            'device': 'auto',
        },
        'annotation': {'lang': 'en'},
        'analysis': {
            'skip_plots': False,
            'top_n_error_types': 10,
        },
        'models': [{
            'name': MODEL_KEY,
            'hf_model_id': LOCAL_MODEL_PATH,
            'model_family': model_info['model_family'],
            'is_learner_tuned': is_learner,
        }],
        'batch_size': BATCH_SIZE,
        'device': 'auto',
        'seed': SEED,
        'output_dir': str(output_dir),
        'skip_plots': False,
    }

    with open(config_path, 'w') as f:
        yaml.dump(config, f, default_flow_style=False)

    return config_path


print(f'Available VRAM: {torch.cuda.mem_get_info()[0] / 1e9:.1f} GB' if device == 'cuda' else 'Running on CPU')

In [ ]:
%%time

all_summaries = {}
all_comparisons = {}
t_total = time.time()

for ds_name, ds_info in DATASETS.items():
    print('\n' + '=' * 70)
    print(f'  DATASET: {ds_name}')
    print(f'  {ds_info["description"]}')
    print('=' * 70)

    if not ds_info['path'].exists():
        print(f'  SKIPPED: file not found at {ds_info["path"]}')
        continue

    output_dir = RUN_DIR / ds_name
    raw_results = output_dir / 'raw_results.json'

    if raw_results.exists():
        print(f'  Already completed (raw_results.json exists). Skipping.')
        continue

    config_path = write_config_yaml(ds_name, ds_info['path'], output_dir)
    print(f'  Config: {config_path}')

    t0 = time.time()
    config = load_config_from_yaml(str(config_path))
    summaries, comparison = run_pipeline(config)
    elapsed = time.time() - t0

    all_summaries[ds_name] = summaries
    all_comparisons[ds_name] = comparison

    print(f'\n  {ds_name} completed in {elapsed / 60:.1f} min')
    for s in summaries:
        ppl = s.get('ppl_mean', 'N/A')
        ppl_str = f'{ppl:.2f}' if isinstance(ppl, (int, float)) else str(ppl)
        avg_err = s.get('avg_errors_per_sentence', 'N/A')
        avg_str = f'{avg_err:.2f}' if isinstance(avg_err, (int, float)) else str(avg_err)
        print(f'    {s["model_name"]}: PPL={ppl_str}, Errors/sent={avg_str}')

    # Free GPU memory between datasets
    if device == 'cuda':
        torch.cuda.empty_cache()

total_elapsed = time.time() - t_total
print(f'\nAll datasets completed in {total_elapsed / 60:.1f} min')

## 3. Cross-Dataset Summary

In [ ]:
import json

cross_summary = {}

for ds_name in DATASETS:
    output_dir = RUN_DIR / ds_name
    entry = {'dataset': ds_name}

    model_summary_path = output_dir / f'{MODEL_KEY}_summary.json'
    baseline_summary_path = output_dir / 'learner_baseline_summary.json'

    if model_summary_path.exists():
        with open(model_summary_path) as f:
            data = json.load(f)
        entry['model'] = {
            'ppl_mean': data.get('ppl_mean'),
            'ppl_median': data.get('ppl_median'),
            'total_errors': data.get('total_errors'),
            'avg_errors_per_sentence': data.get('avg_errors_per_sentence'),
            'error_rate': data.get('error_rate'),
            'top_error_types': data.get('top_10_error_types', [])[:5],
        }

    if baseline_summary_path.exists():
        with open(baseline_summary_path) as f:
            data = json.load(f)
        entry['learner_baseline'] = {
            'total_errors': data.get('total_errors'),
            'avg_errors_per_sentence': data.get('avg_errors_per_sentence'),
            'error_rate': data.get('error_rate'),
            'top_error_types': data.get('top_10_error_types', [])[:5],
        }

    cross_summary[ds_name] = entry

# Print summary table
print(f'Cross-Dataset Summary for {MODEL_KEY} ({model_info["params"]})')
print(f'{"Dataset":<25} {"PPL Mean":>10} {"Errors/Sent":>12} {"Error Rate":>12}')
print(f'{"-"*25} {"-"*10} {"-"*12} {"-"*12}')

for name, data in cross_summary.items():
    m = data.get('model', {})
    ppl = m.get('ppl_mean', 'N/A')
    avg_err = m.get('avg_errors_per_sentence', 'N/A')
    err_rate = m.get('error_rate', 'N/A')
    ppl_s = f'{ppl:.2f}' if isinstance(ppl, (int, float)) else str(ppl)
    avg_s = f'{avg_err:.2f}' if isinstance(avg_err, (int, float)) else str(avg_err)
    rate_s = f'{err_rate:.3f}' if isinstance(err_rate, (int, float)) else str(err_rate)
    print(f'{name:<25} {ppl_s:>10} {avg_s:>12} {rate_s:>12}')

# Compare with learner baseline
print(f'\nLearner Baseline Comparison:')
print(f'{"Dataset":<25} {"Model Err/Sent":>14} {"Learner Err/Sent":>16} {"Ratio":>8}')
print(f'{"-"*25} {"-"*14} {"-"*16} {"-"*8}')

for name, data in cross_summary.items():
    m_err = data.get('model', {}).get('avg_errors_per_sentence')
    l_err = data.get('learner_baseline', {}).get('avg_errors_per_sentence')
    if m_err and l_err and l_err > 0:
        print(f'{name:<25} {m_err:>14.2f} {l_err:>16.2f} {m_err/l_err:>8.2f}')
    else:
        print(f'{name:<25} {"N/A":>14} {"N/A":>16} {"N/A":>8}')

## 4. Visualize Results

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Error Rate Comparison (Model vs Learner Baseline) ──────────────────

ds_names = []
model_rates = []
learner_rates = []

ds_display = {
    'norm-CELVA-SP': 'CELVA-SP',
    'norm-EFCAMDAT-test': 'EFCAMDAT-test',
    'norm-KUPA-KEYS': 'KUPA-KEYS',
}

for name, data in cross_summary.items():
    m_err = data.get('model', {}).get('avg_errors_per_sentence')
    l_err = data.get('learner_baseline', {}).get('avg_errors_per_sentence')
    if m_err is not None and l_err is not None:
        ds_names.append(ds_display.get(name, name))
        model_rates.append(m_err)
        learner_rates.append(l_err)

if ds_names:
    x = np.arange(len(ds_names))
    width = 0.35

    fig, ax = plt.subplots(figsize=(10, 5))
    bars1 = ax.bar(x - width/2, model_rates, width, label=MODEL_KEY, color='#4C72B0')
    bars2 = ax.bar(x + width/2, learner_rates, width, label='Learner baseline', color='#DD8452')

    ax.set_ylabel('Avg errors per sentence')
    ax.set_title(f'Error Profile: {MODEL_KEY} vs Learner Baseline')
    ax.set_xticks(x)
    ax.set_xticklabels(ds_names)
    ax.legend()
    ax.bar_label(bars1, fmt='%.2f', padding=3)
    ax.bar_label(bars2, fmt='%.2f', padding=3)

    plt.tight_layout()
    fig.savefig(str(RUN_DIR / 'error_comparison.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {RUN_DIR / "error_comparison.png"}')
else:
    print('No results to plot.')

In [ ]:
# ── Top Error Types per Dataset ───────────────────────────────────────

n_plots = sum(1 for d in cross_summary.values() if 'model' in d and d['model'].get('top_error_types'))

if n_plots > 0:
    fig, axes = plt.subplots(1, n_plots, figsize=(6 * n_plots, 5))
    if n_plots == 1:
        axes = [axes]

    plot_idx = 0
    for ds_name, data in cross_summary.items():
        model_data = data.get('model', {})
        top_types = model_data.get('top_error_types', [])
        if not top_types:
            continue

        labels = [t[0] if isinstance(t, (list, tuple)) else t.get('type', '?') for t in top_types]
        counts = [t[1] if isinstance(t, (list, tuple)) else t.get('count', 0) for t in top_types]

        ax = axes[plot_idx]
        ax.barh(labels[::-1], counts[::-1], color='#4C72B0')
        ax.set_xlabel('Count')
        ax.set_title(ds_display.get(ds_name, ds_name))
        plot_idx += 1

    fig.suptitle(f'Top Error Types: {MODEL_KEY}', fontsize=14, y=1.02)
    plt.tight_layout()
    fig.savefig(str(RUN_DIR / 'top_error_types.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {RUN_DIR / "top_error_types.png"}')
else:
    print('No error type data to plot.')

## 5. Save Everything to Google Drive

In [ ]:
# Save cross-dataset summary
summary_path = RUN_DIR / 'cross_dataset_summary.json'
with open(summary_path, 'w') as f:
    json.dump(cross_summary, f, indent=2, default=str)
print(f'Cross-dataset summary: {summary_path}')

# Save run configuration
run_config = {
    'model_key': MODEL_KEY,
    'model_info': {
        'params': model_info['params'],
        'model_family': model_info['model_family'],
        'is_learner_tuned': model_info.get('is_learner_tuned', False),
        'description': model_info['description'],
        'model_path': MODEL_PATH,
    },
    'pipeline': {
        'gec_model': GEC_MODEL,
        'gec_method': GEC_METHOD,
        'batch_size': BATCH_SIZE,
        'gec_batch_size': GEC_BATCH_SIZE,
        'seed': SEED,
        'max_sentences': MAX_SENTENCES,
    },
    'datasets': {name: str(info['path']) for name, info in DATASETS.items()},
    'timestamp': TIMESTAMP,
    'device': device,
    'gpu': torch.cuda.get_device_name() if device == 'cuda' else None,
}

config_path = RUN_DIR / 'run_config.yaml'
with open(config_path, 'w') as f:
    yaml.dump(run_config, f, default_flow_style=False)
print(f'Run config: {config_path}')

# List all output files
print(f'\n=== All artifacts saved to {RUN_DIR} ===')
for p in sorted(RUN_DIR.rglob('*')):
    if p.is_file():
        size_mb = p.stat().st_size / 1e6
        rel = p.relative_to(RUN_DIR)
        print(f'  {rel}  ({size_mb:.1f} MB)' if size_mb > 0.1 else f'  {rel}')